# PA3 · Task 1 · Evaluation, diagnostics, qualitative behaviour

**What this notebook does.** Evaluates the 7 conditions (baseline, Full FT midpoint, 5 final models) on GSM8K / HellaSwag / BoolQ / ARC-Easy with lm-evaluation-harness, generates the 168 qualitative responses, runs the weight-space and activation diagnostics, and builds every Task 1 table and figure, plus the 42-response annotation sheet.

**Settings:** **GPU T4 ×2**, Internet **On**. Roughly 2–3 h.
**Attach:** `pa3-task1` and the output of the **last** `t1_train` session (it contains all five finished runs). If this notebook stops early, run it again with its own output attached as well: finished conditions, and finished stages within a condition (GSM8K / likelihood tasks / qualitative), are reused.

**How to read the output:** `PASS / FAIL / OBSERVE`, then the **expected** and **actual** lines.

In [ ]:
# S1. Find the Task 1 code (attached Kaggle dataset, or a clone of your GitHub repo), copy it to /tmp, check the GPUs.
# All GPU work runs in separate processes, so this notebook itself never holds GPU memory.
import os, sys, glob, json, time, shutil, subprocess
from pathlib import Path
SESSION_START = time.time()     # Kaggle's 12-hour limit counts from about now

REPO_URL = ""        # optional: your public GitHub repo URL, used only if no code dataset is attached
REQUIRED_CODE_VERSION = "2026.10.30.2" # this notebook needs this code version or newer
CODE = Path("/tmp/pa3_task1_code")

def code_version(root):
    import re
    try:
        m = re.search(r'CODE_VERSION\s*=\s*"([^"]+)"', (Path(root) / "common" / "__init__.py").read_text())
        return m.group(1) if m else "0 (no version stamp: older than 2026.10.30)"
    except FileNotFoundError:
        return "0 (no version stamp: older than 2026.10.30)"

# every attached copy of the code, newest version first (a dataset can hold several copies after re-uploads)
hits = sorted({Path(p).parents[1] for p in glob.glob("/kaggle/input/**/task1/t1_train.py", recursive=True)},
              key=code_version, reverse=True)
for h in hits:
    print(f"found code copy: {h}  (version {code_version(h)})")
if CODE.exists():
    shutil.rmtree(CODE)
if hits:
    if code_version(hits[0]) < REQUIRED_CODE_VERSION:
        raise SystemExit(f"The newest attached code is version {code_version(hits[0])}, but this notebook needs "
                         f"{REQUIRED_CODE_VERSION}. Upload the latest pa3-task1.zip as a new dataset version and re-attach it.")
    shutil.copytree(hits[0], CODE)
    print("using:", hits[0])
elif REPO_URL:
    subprocess.run(["git", "clone", "-q", REPO_URL, str(CODE)], check=True)
    print("code cloned from", REPO_URL)
else:
    raise SystemExit("Attach the pa3-task1 dataset (or set REPO_URL) and re-run this cell.")
sys.path.insert(0, str(CODE))
os.chdir(CODE)
CODE_COPIES = [str(h) for h in hits]
def inputs(pattern):
    """Files matching `pattern` in the ATTACHED OUTPUTS only: earlier runs and results are always read from the original
    notebook output you attached, never from a copy inside the code dataset."""
    return [p for p in glob.glob(pattern, recursive=True) if not any(p.startswith(c + "/") for c in CODE_COPIES)]
from common.utils import Checks, human_bytes, dir_size_bytes, read_json, read_jsonl
CHK = Checks()
CHK.check(code_version(CODE) >= REQUIRED_CODE_VERSION, "code version in use", f">= {REQUIRED_CODE_VERSION}", code_version(CODE))
try:
    smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
    GPUS = [l.strip() for l in smi.stdout.strip().splitlines() if l.strip()]
except FileNotFoundError:
    GPUS = []
CHK.check(len(GPUS) >= 1, "GPU visible", "GPU T4 x2 (Settings -> Accelerator)", GPUS or "no GPU: is the accelerator switched on?")
import importlib.metadata as md
vers = {p: md.version(p) for p in ["torch", "transformers", "datasets"]}
CHK.observe("library versions", vers)
tv = tuple(int(x) for x in vers["transformers"].split(".")[:2])
if tv < (4, 51):     # Qwen3 support arrived in transformers 4.51
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers"], check=True)
    print("upgraded transformers: restart the kernel and run again")

In [ ]:
# S2. Install the evaluation harness (pinned version) and check that every training run is attached.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lm-eval==0.4.13", "openpyxl"], check=True)
from task1.t1_eval import find_run, ORDER
WORK = Path("/kaggle/working"); EVAL = WORK / "t1_eval"
for m in ["full", "l2sp", "lora4", "lora16", "oplora"]:
    r = find_run(m)
    CHK.check(r is not None and (r / "DONE").exists(), f"training run '{m}' attached and finished", "t1_runs/" + m, r)
probe = inputs("/kaggle/input/**/t1_data/t1_probe_set.json")
CHK.check(bool(probe), "probe set attached (from t1_data)", "t1_probe_set.json", probe[:1])
PROBE = probe[0]
for c in ORDER:     # bring over whatever an earlier attempt of this notebook finished (whole conditions or single stages)
    stages = ["gsm8k_part.json", "mc_part.json", "qual_responses.jsonl", "metrics.json"]
    prev = sorted({Path(p).parent for st in stages for p in inputs(f"/kaggle/input/**/t1_eval/{c}/{st}")},
                  key=lambda d: sum((d / st).exists() for st in stages), reverse=True)
    if prev and not (EVAL / c).exists():
        shutil.copytree(prev[0], EVAL / c)
        print(f"{c}: reused from {prev[0]}: {[st for st in stages if (EVAL / c / st).exists()]}")

In [ ]:
# S3. Launch evaluation workers (benchmarks + 24 qualitative prompts per condition), one per GPU.
SPLIT = {0: ["baseline", "full_mid", "full_final", "l2sp_final"], 1: ["lora4_final", "lora16_final", "oplora_final"]}
if len(GPUS) < 2:
    SPLIT = {0: ORDER}
procs = {}
for gpu, conds in SPLIT.items():
    cmd = f"python -u -m task1.t1_eval --conditions {' '.join(conds)} --out {EVAL} --probe {PROBE} >> {WORK}/t1_eval_gpu{gpu}.log 2>&1"
    procs[gpu] = subprocess.Popen(["bash", "-c", cmd], cwd=CODE, env={**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu),
                                                                    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"})
    print(f"GPU {gpu}: {conds}")

In [ ]:
# S4. Monitor until both workers finish.
def tail(path, n=2):
    try:
        return "".join([l for l in open(path) if l.strip() and "it/s]" not in l and "%|" not in l][-n:]).strip()
    except FileNotFoundError:
        return "(not started)"
t0 = time.time()
while any(p.poll() is None for p in procs.values()):
    time.sleep(300)
    print(f"--- {(time.time() - t0) / 60:.0f} min")
    for g in procs: print(f"  GPU{g}: {tail(WORK / f't1_eval_gpu{g}.log')}")
for g, p in procs.items():
    CHK.check(p.returncode == 0, f"evaluation worker GPU {g} exited cleanly", 0, p.returncode)
    if p.returncode: print(tail(WORK / f"t1_eval_gpu{g}.log", 40))

In [ ]:
# S5. Intended vs actual for each condition.
from task1.t1_eval import EXPECTED_N
ALL_DONE = all((EVAL / c / "DONE").exists() for c in ORDER)
for c in ORDER:
    d = EVAL / c
    if not (d / "metrics.json").exists():
        done = [st for st in ["gsm8k_part.json", "mc_part.json", "qual_responses.jsonl"] if (d / st).exists()]
        CHK.check(False, f"{c}: metrics present", "metrics.json", f"missing (stages saved so far: {done or 'none'})"); continue
    m = read_json(d / "metrics.json")
    print(f"\n======== {c} ========")
    CHK.check(m["n_examples"] == EXPECTED_N, f"{c}: complete splits evaluated", "1319 / 10042 / 3270 / 2376", m["n_examples"])
    CHK.check(m["gsm8k_tracked_generations"] == 1319, f"{c}: every GSM8K generation has an exact stop reason", 1319, m["gsm8k_tracked_generations"])
    if c == "baseline":
        print("first GSM8K prompt exactly as the model saw it:\n" + repr(m["first_gsm8k_prompt"]))
        CHK.check(m["first_gsm8k_prompt"].endswith("<think>\n\n</think>\n\n"), "GSM8K prompt uses the non-thinking chat template (empty think block)",
                  "ends with '<think>\\n\\n</think>\\n\\n'", repr(m["first_gsm8k_prompt"][-25:]))
        CHK.check(m["lm_eval_config"]["gsm8k"]["num_fewshot"] == 0, "GSM8K is zero-shot", 0, m["lm_eval_config"]["gsm8k"]["num_fewshot"])
        CHK.observe("GSM8K generation settings used", m["lm_eval_config"]["gsm8k"]["generation_kwargs"])
    if "merge_check" in m:
        mc = m["merge_check"]
        CHK.check(mc["argmax_agreement"] > 0.98, f"{c}: merged vs unmerged logits agree", "argmax agreement > 0.98, small |diff| (FP16 rounding)",
                  f"agreement {mc['argmax_agreement']:.4f}, max |diff| {mc['max_abs_logit_diff']:.3f}, mean {mc['mean_abs_logit_diff']:.4f}")
    CHK.observe(f"{c}: primary scores (%)", {k: round(100 * m[k], 2) for k in ["gsm8k_flexible_em", "hellaswag_acc_norm", "boolq_acc", "arc_easy_acc_norm"]})
    CHK.observe(f"{c}: batch sizes used (GSM8K / likelihood tasks)", (m.get("gsm8k_batch_size"), m.get("mc_batch_size")))
    CHK.observe(f"{c}: GSM8K stop reasons / unextractable / mean tokens",
                (m["gsm8k_stop_reasons"], f"{100 * m['gsm8k_unextractable_rate']:.1f}%", round(m["gsm8k_mean_gen_tokens"], 1)))
    q = read_jsonl(d / "qual_responses.jsonl") if (d / "qual_responses.jsonl").exists() else []
    CHK.check(len(q) == 24, f"{c}: 24 qualitative responses saved", 24, len(q))

In [ ]:
# S6. The probe prompts must match lm-eval's prompts exactly; then the diagnostics (down_proj, layers 0/13/27, FP32).
if not ALL_DONE:
    print("SKIPPED: not every condition finished (see S4/S5). Run this notebook again with THIS version's output attached:"
          " finished conditions and stages are reused.")
else:
  from transformers import AutoTokenizer
  from task1.t1_diag import check_probe_formats
  tok = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
  check_probe_formats(tok, read_json(PROBE), EVAL / "baseline", CHK)
  r = subprocess.run([sys.executable, "-m", "task1.t1_diag", "--probe", PROBE, "--out", str(WORK / "t1_diag")], cwd=CODE,
                   capture_output=True, text=True, env={**os.environ, "CUDA_VISIBLE_DEVICES": "0"})
  print(r.stdout[-12000:]); print(r.stderr[-3000:] if r.returncode else "")
  CHK.check(r.returncode == 0 and "0 FAIL" in r.stdout, "diagnostics finished and their own checks passed", "0 FAIL", r.stdout.strip().splitlines()[-1] if r.stdout.strip() else r.returncode)

In [ ]:
# S7. Tables, paired tests, figures and the 42-response annotation sheet.
if not ALL_DONE:
    print("SKIPPED: evaluation incomplete (see S5).")
else:
    from task1 import t1_report as R
    import pandas as pd
    pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
    REP = WORK / "t1_report"; REP.mkdir(exist_ok=True)
    bench, comp, rows = R.benchmark_table(REP)
    print(bench.round(2).to_string(index=False)); print(); print(comp.round(2).to_string(index=False)); print()
    print(R.paired_tests(REP).round(4).to_string(index=False)); print()
    R.training_figures(REP)
    print(R.resource_table(REP).to_string(index=False))
    R.diag_figure(REP, WORK / "t1_diag")
    sub, summ, (first, last) = R.annotation_sheet(REP)
    CHK.check(len(sub) == 42, "annotation subset = first + last question x 3 formats x 7 conditions", 42, len(sub))
    print(f"\nannotate '{first}' and '{last}'  ->  t1_report/t1_annotation_42.xlsx"); print(summ.round(3).to_string())
    for a, b in [("baseline", "full_final"), ("baseline", "lora16_final"), ("lora16_final", "oplora_final"), ("full_mid", "full_final")]:
        CHK.observe(f"GSM8K per-question transitions {a} -> {b}", R.gsm8k_transitions(a, b))

In [ ]:
# S8. Look at the figures, then one aligned comparison (change QUESTION / FORMAT freely).
if not ALL_DONE:
    print("SKIPPED: evaluation incomplete (see S5).")
else:
    from IPython.display import Image, display
    for f in ["t1_loss_lr.png", "t1_l2sp.png", "t1_diagnostics.png"]:
        if (REP / f).exists(): display(Image(str(REP / f)))
    QUESTION, FORMAT = first, "json"
    R.aligned(QUESTION, FORMAT)

In [ ]:
# S9. Machine-readable results for the GitHub repo (no model weights): t1_results.zip in the output.
import zipfile
keep_ext = {".json", ".jsonl", ".csv", ".png", ".xlsx", ".log", ".yaml"}
roots = [WORK / "t1_eval", WORK / "t1_diag", WORK / "t1_report"] + [find_run(m) for m in ["full", "l2sp", "lora4", "lora16", "oplora"]]
data_dir = Path(PROBE).parent
with zipfile.ZipFile(WORK / "t1_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in [data_dir / "t1_train_ids.json", data_dir / "t1_probe_set.json"]:
        z.write(f, f"t1_data/{f.name}")
    for root in [r for r in roots if r is not None and Path(r).exists()]:
        for f in Path(root).rglob("*"):
            if f.is_file() and f.suffix in keep_ext and "recovery_u" not in str(f) and "export_" not in str(f):
                arc = f"t1_runs/{root.name}/{f.relative_to(root)}" if root.parent.name == "t1_runs" else f"{root.name}/{f.relative_to(root)}"
                z.write(f, arc)
CHK.observe("t1_results.zip", human_bytes((WORK / "t1_results.zip").stat().st_size), note="download it and commit its contents under results/task1/")
CHK.summary(); CHK.save(WORK / "t1_eval_checks.json")
print("\nNEXT:", "Task 1 evaluation complete: send S5-S8 output and t1_results.zip." if ALL_DONE else
      "not finished. Run again with pa3-task1, the training output AND this version's output attached; finished stages are reused.")